In [2]:
import duckdb, glob, os
con = duckdb.connect()
for f in glob.glob('data/raw/*.csv'):
    name = os.path.basename(f).replace('olist_','').replace('_dataset.csv','').replace('.csv','')
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv_auto('{f}')")
con.sql("SHOW TABLES").show()

┌───────────────────────────────────┐
│               name                │
│              varchar              │
├───────────────────────────────────┤
│ customers                         │
│ geolocation                       │
│ order_items                       │
│ order_payments                    │
│ order_reviews                     │
│ orders                            │
│ product_category_name_translation │
│ products                          │
│ sellers                           │
└───────────────────────────────────┘



In [3]:
con.sql("DESCRIBE orders").show()

┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_id                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ order_status                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ order_purchase_timestamp      │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_approved_at             │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_delivered_carrier_date  │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_delivered_customer_date │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_estimated_delivery_date 

In [28]:
print("Orders BY status count")
con.sql("""
    SELECT order_status, COUNT(*) AS status_count
    FROM orders
    GROUP BY order_status
    ORDER BY order_status
""").show()

Orders BY status count
┌──────────────┬──────────────┐
│ order_status │ status_count │
│   varchar    │    int64     │
├──────────────┼──────────────┤
│ approved     │            2 │
│ canceled     │          625 │
│ created      │            5 │
│ delivered    │        96478 │
│ invoiced     │          314 │
│ processing   │          301 │
│ shipped      │         1107 │
│ unavailable  │          609 │
└──────────────┴──────────────┘



In [10]:
con.sql("DESCRIBE order_items").show()

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ order_item_id       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ seller_id           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ shipping_limit_date │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ price               │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ freight_value       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [29]:
print("Monthly order count with Total Revenue")
con.sql("""
    SELECT DATE_TRUNC('month', order_purchase_timestamp):: DATE `MONTH`, 
    COUNT(DISTINCT order_id) `MONTHLY_COUNT`, SUM(price) `TOTAL_REVENUE` 
    FROM orders 
    JOIN order_items 
    USING(order_id) 
    WHERE order_status = 'delivered' 
    GROUP BY 1 
    ORDER BY 1
""").show(max_rows=30)

Monthly order count with Total Revenue
┌────────────┬───────────────┬────────────────────┐
│   MONTH    │ MONTHLY_COUNT │   TOTAL_REVENUE    │
│    date    │     int64     │       double       │
├────────────┼───────────────┼────────────────────┤
│ 2016-09-01 │             1 │             134.97 │
│ 2016-10-01 │           265 │  40325.11000000003 │
│ 2016-12-01 │             1 │               10.9 │
│ 2017-01-01 │           750 │  111798.3600000003 │
│ 2017-02-01 │          1653 │ 234223.39999999932 │
│ 2017-03-01 │          2546 │  359198.8499999958 │
│ 2017-04-01 │          2303 │ 340669.67999999656 │
│ 2017-05-01 │          3546 │  489338.2499999915 │
│ 2017-06-01 │          3135 │ 421923.36999999365 │
│ 2017-07-01 │          3872 │  481604.5199999908 │
│ 2017-08-01 │          4193 │  554699.6999999915 │
│ 2017-09-01 │          4150 │   607399.669999995 │
│ 2017-10-01 │          4478 │  648247.6499999976 │
│ 2017-11-01 │          7289 │  987765.3700000149 │
│ 2017-12-01 │          5

In [18]:
print("Total delivered orders count")
con.sql("""
SELECT COUNT(*) FROM orders WHERE order_status = 'delivered'
""").show()

Total delivered orders count
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        96478 │
└──────────────┘



In [24]:
print("Total delivered orders count ")
con.sql("""
SELECT SUM(MONTHLY_COUNT) FROM (
    SELECT DATE_TRUNC('month', order_purchase_timestamp):: DATE `MONTH`, COUNT(DISTINCT order_id) `MONTHLY_COUNT`, SUM(price) `TOTAL_REVENUE` FROM orders JOIN order_items USING(order_id) WHERE order_status = 'delivered' GROUP BY 1
)
""").show()

Total delivered orders count 
┌────────────────────┐
│ sum(MONTHLY_COUNT) │
│       int128       │
├────────────────────┤
│              96478 │
└────────────────────┘



In [31]:
print("Average order value per month")
con.sql("""
SELECT DATE_TRUNC('month', order_purchase_timestamp):: DATE `MONTH`,
    SUM(price) `TOTAL_REVENUE`,
    COUNT(DISTINCT order_id) `MONTHLY_COUNT`,
    SUM(price)/COUNT(DISTINCT order_id) `AVERAGE_ORDER_VALUE`
FROM orders
JOIN order_items USING(order_id)
WHERE order_status = 'delivered'
GROUP BY 1
ORDER BY 1
""").show(max_rows=30)

Average order value per month
┌────────────┬────────────────────┬───────────────┬─────────────────────┐
│   MONTH    │   TOTAL_REVENUE    │ MONTHLY_COUNT │ AVERAGE_ORDER_VALUE │
│    date    │       double       │     int64     │       double        │
├────────────┼────────────────────┼───────────────┼─────────────────────┤
│ 2016-09-01 │             134.97 │             1 │              134.97 │
│ 2016-10-01 │  40325.11000000003 │           265 │  152.17022641509445 │
│ 2016-12-01 │               10.9 │             1 │                10.9 │
│ 2017-01-01 │  111798.3600000003 │           750 │   149.0644800000004 │
│ 2017-02-01 │ 234223.39999999932 │          1653 │  141.69594676345997 │
│ 2017-03-01 │  359198.8499999958 │          2546 │  141.08360172819945 │
│ 2017-04-01 │ 340669.67999999656 │          2303 │   147.9243074250962 │
│ 2017-05-01 │  489338.2499999915 │          3546 │  137.99725042300946 │
│ 2017-06-01 │ 421923.36999999365 │          3135 │  134.58480701754183 │
│ 2017-0

In [32]:
print("Percentage of orders delivered after estimated delivery date")
con.sql("""
SELECT
    COUNT(*) AS total_orders,
    SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END) AS late_deliveries,
    (SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS percentage_late_deliveries
FROM orders
WHERE order_status = 'delivered'
""").show()

Percentage of orders delivered after estimated delivery date
┌──────────────┬─────────────────┬────────────────────────────┐
│ total_orders │ late_deliveries │ percentage_late_deliveries │
│    int64     │     int128      │           double           │
├──────────────┼─────────────────┼────────────────────────────┤
│        96478 │            7826 │          8.111693857667033 │
└──────────────┴─────────────────┴────────────────────────────┘

